# Phase 6: 構造化RAGシステム実験

**目的**: 座標計算・集計・比較機能を統合した構造化RAGシステムの評価

**メモリ最適化版**: LLMロード前にEmbeddingを解放

**作成日**: 2026-01-22

## Section 1: 環境セットアップ

In [ ]:
# 1.1 パッケージインストール
%%capture
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma
!pip install -q chromadb sentence-transformers
!pip install -q tqdm pandas

print("パッケージインストール完了")

In [ ]:
# 1.2 GPU・メモリ確認
import torch
import gc

def print_memory_status():
    """メモリ状況を表示"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"GPU VRAM: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
    
    import psutil
    ram = psutil.virtual_memory()
    print(f"RAM: {ram.used/1e9:.2f}GB / {ram.total/1e9:.2f}GB ({ram.percent}%)")

def clear_memory():
    """メモリをクリア"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("メモリクリア完了")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_memory:.1f}GB)")
else:
    print("GPUが利用できません")

print_memory_status()

In [ ]:
# 1.3 Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.4 パス設定
import os
import sys

BASE_DIR = "/content/drive/MyDrive/experiments-local-llm"
DATA_DIR = f"{BASE_DIR}/data"
RESULTS_DIR = f"{BASE_DIR}/results"
CHROMA_DIR = f"{BASE_DIR}/chroma_db"

for dir_path in [DATA_DIR, RESULTS_DIR, CHROMA_DIR]:
    os.makedirs(dir_path, exist_ok=True)

sys.path.insert(0, BASE_DIR)
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# 1.5 モデル設定
USE_SMALL_MODEL = False  # Trueで3Bモデル使用

if USE_SMALL_MODEL:
    LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
else:
    LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"

EMBEDDING_MODEL = "intfloat/multilingual-e5-base"

print(f"LLM: {LLM_MODEL}")
print(f"Embedding: {EMBEDDING_MODEL}")

## Section 2: データ読み込み

In [ ]:
# 2.1 POIデータ読み込み
import json
from collections import Counter

poi_path = f"{DATA_DIR}/poi_documents.json"

with open(poi_path, "r", encoding="utf-8") as f:
    poi_documents = json.load(f)

all_pois = []
for doc in poi_documents:
    if "metadata" in doc:
        poi = doc["metadata"].copy()
        poi["content"] = doc.get("content", "")
    else:
        poi = doc.copy()
    all_pois.append(poi)

print(f"POIデータ: {len(all_pois)}件")
print_memory_status()

## Section 3: Phase 6モジュールテスト（LLM不要）

In [ ]:
# 3.1 geo_utilsテスト
from src.geo_utils import SHIBUYA_STATION, enrich_all_pois

print(f"基準点: {SHIBUYA_STATION['name']}")

enriched_pois = enrich_all_pois(all_pois)
print(f"空間情報追加完了: {len(enriched_pois)}件")

direction_counts = Counter(poi.get("direction_from_station", "不明") for poi in enriched_pois)
print("\n方向別分布:")
for d, c in direction_counts.most_common():
    print(f"  {d}: {c}件")

In [ ]:
# 3.2 aggregatorテスト
from src.aggregator import compare_east_west, get_top_categories

ew_all = compare_east_west(enriched_pois)
print(f"【東西比較（全体）】\n  {ew_all.to_japanese()}")

ew_cafe = compare_east_west(enriched_pois, "カフェ")
print(f"\n【東西比較（カフェ）】\n  {ew_cafe.to_japanese()}")

print("\n【カテゴリランキング TOP5】")
for i, cat in enumerate(get_top_categories(enriched_pois, 5), 1):
    print(f"  {i}. {cat.category}: {cat.count}件")

In [ ]:
# 3.3 質問分析テスト
from src.structured_rag_system import analyze_question

test_questions = [
    "渋谷駅の東側と西側、どちらにカフェが多いですか？",
    "渋谷駅周辺で最も多いPOIカテゴリは？",
    "渋谷駅から500m以内のコンビニを教えて"
]

print("【質問分析テスト】")
for q in test_questions:
    a = analyze_question(q)
    print(f"\n{q}")
    print(f"  → タイプ: {a.question_type}, カテゴリ: {a.subcategories}")

## Section 4: ベクトルストア構築（永続化対応）

In [ ]:
# 4.1 永続化確認
COLLECTION_NAME = "poi_shibuya_phase6"
chroma_exists = os.path.exists(f"{CHROMA_DIR}/{COLLECTION_NAME}")
print(f"永続化ベクトルストア: {'存在' if chroma_exists else '未作成'}")

In [ ]:
# 4.2 Embeddingモデルロード
from langchain_huggingface import HuggingFaceEmbeddings

print(f"Embeddingモデルロード中: {EMBEDDING_MODEL}")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
print("Embeddingモデルロード完了")
print_memory_status()

In [ ]:
# 4.3 ベクトルストア構築または読み込み
from langchain_chroma import Chroma
from langchain_core.documents import Document

if chroma_exists:
    print("既存のベクトルストアを読み込み中...")
    vectorstore = Chroma(
        persist_directory=CHROMA_DIR,
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings
    )
else:
    print("ベクトルストア新規構築中...")
    documents = []
    for poi in poi_documents:
        if "metadata" in poi:
            documents.append(Document(page_content=poi["content"], metadata=poi["metadata"]))
        else:
            content = poi.get("content", f"{poi.get('name', '')} - {poi.get('category', '')}")
            documents.append(Document(page_content=content, metadata=poi))
    
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DIR
    )
    print(f"ベクトルストア構築完了: {len(documents)}件")

print_memory_status()

In [ ]:
# 4.4 Embeddingモデルを解放
print("Embeddingモデルを解放してメモリを確保...")
del embeddings
clear_memory()
print_memory_status()

## Section 5: LLMモデルロード

In [ ]:
# 5.1 LLMモデルロード
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print(f"LLMモデルロード中: {LLM_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
print("LLMモデルロード完了")
print_memory_status()

## Section 6: 構造化RAGシステム（軽量版）

In [ ]:
# 6.1 軽量版RAGシステム
from src.aggregator import analyze_category_by_direction

class LightweightStructuredRAG:
    def __init__(self, model, tokenizer, vectorstore, all_pois):
        self.model = model
        self.tokenizer = tokenizer
        self.vectorstore = vectorstore
        self.all_pois = all_pois
        self.system_prompt = "あなたは渋谷エリアの地理情報に詳しいアシスタントです。提供された情報に基づいて、正確かつ簡潔に回答してください。"
    
    def _generate(self, prompt, max_tokens=256):
        messages = [{"role": "system", "content": self.system_prompt}, {"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.1, do_sample=True, pad_token_id=self.tokenizer.eos_token_id)
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "assistant" in response.lower():
            response = response.split("assistant")[-1].strip()
        return response
    
    def query(self, question):
        import time
        start = time.time()
        
        analysis = analyze_question(question)
        context_parts = []
        
        if analysis.requires_comparison and "東" in question and "西" in question:
            cat = analysis.subcategories[0] if analysis.subcategories else None
            result = compare_east_west(self.all_pois, cat)
            context_parts.append(f"【東西比較】\n{result.to_japanese()}")
            if cat:
                detail = analyze_category_by_direction(self.all_pois, cat)
                context_parts.append(f"\n詳細:")
                for d, c in detail['by_direction'].items():
                    context_parts.append(f"  {d}: {c}件")
        elif analysis.requires_aggregation:
            top = get_top_categories(self.all_pois, 5)
            context_parts.append("【カテゴリランキング】")
            for i, cat in enumerate(top, 1):
                context_parts.append(f"  {i}. {cat.category}: {cat.count}件")
        
        try:
            results = self.vectorstore.similarity_search(question, k=3)
            if results:
                context_parts.append("\n【関連POI】")
                for r in results[:3]:
                    context_parts.append(f"  - {r.metadata.get('name', '不明')}")
        except Exception as e:
            print(f"ベクトル検索スキップ: {e}")
        
        context = "\n".join(context_parts)
        prompt = f"以下の情報を参考に回答してください。\n\n{context}\n\n質問: {question}\n回答:"
        
        answer = self._generate(prompt)
        elapsed = time.time() - start
        
        return {"answer": answer, "analysis": analysis.to_dict(), "context": context, "time_sec": round(elapsed, 2)}

rag = LightweightStructuredRAG(model, tokenizer, vectorstore, enriched_pois)
print("軽量版構造化RAGシステム初期化完了")
print_memory_status()

## Section 7: テスト実行

In [ ]:
# 7.1 テスト実行
test_questions = [
    "渋谷駅の東側と西側、どちらにカフェが多いですか？",
    "渋谷駅周辺で最も多いPOIカテゴリは何ですか？上位3つを教えてください"
]

print("=" * 70)
print("構造化RAGテスト")
print("=" * 70)

for q in test_questions:
    print(f"\n質問: {q}")
    print("-" * 50)
    
    result = rag.query(q)
    
    print(f"分析: {result['analysis']['question_type']}")
    print(f"時間: {result['time_sec']}秒")
    print(f"\n回答:\n{result['answer'][:600]}")
    print("=" * 70)
    clear_memory()

## Section 8: 結果保存

In [ ]:
# 8.1 結果保存
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

result_data = {
    "timestamp": timestamp,
    "phase": "Phase 6 - Structured RAG",
    "model": LLM_MODEL,
    "poi_count": len(enriched_pois),
    "direction_distribution": dict(direction_counts),
    "east_west_comparison": compare_east_west(enriched_pois).to_dict()
}

output_path = f"{RESULTS_DIR}/phase6_result_{timestamp}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result_data, f, ensure_ascii=False, indent=2)

print(f"結果保存: {output_path}")
print_memory_status()